# Kaggriculture's mean vote beats Biohub's by 52%

A companion notebook of mine, [The Real Median Kaggle Notebook Gets 2
Votes](https://www.kaggle.com/code/dariushafshar/the-real-median-kaggle-notebook-gets-2-votes),
showed that "venue median" numbers people quote (mine included) are usually the median of a
venue's top 20 most-voted notebooks, not the median notebook -- and the true population median
is close to flat everywhere.

That fixed one bug. It left a second one sitting in the same statistic, and this notebook is about
that one: **"top 20" is a fixed count, not a fixed percentile, and competitions don't have the
same number of notebooks.** Comparing top-20 medians across venues of different sizes compares
different percentiles without saying so.

## Load every kernel and attribute it to its current competition source

In [ ]:
import glob, time
t0 = time.time()

def find(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"{pattern} not found under /kaggle/input -- is kaggle/meta-kaggle mounted?"
    return hits[0]

KERNELS_PATH = find("Kernels.csv")
KVCS_PATH = find("KernelVersionCompetitionSources.csv")
COMPS_PATH = find("Competitions.csv")
print(f"reading {KERNELS_PATH}\n reading {KVCS_PATH}\n reading {COMPS_PATH}")

In [ ]:
import pandas as pd
import numpy as np

kernels = pd.read_csv(KERNELS_PATH, usecols=["Id", "CurrentKernelVersionId", "TotalViews", "TotalVotes"])
kvcs = pd.read_csv(KVCS_PATH, usecols=["KernelVersionId", "SourceCompetitionId"])
comps = pd.read_csv(COMPS_PATH, usecols=["Id", "Slug", "Title", "TotalTeams"])

merged = kernels.merge(kvcs, left_on="CurrentKernelVersionId", right_on="KernelVersionId", how="inner")
merged = merged.merge(comps, left_on="SourceCompetitionId", right_on="Id", how="left", suffixes=("", "_comp"))
print(f"{len(merged):,} competition-attributed kernels, {merged['SourceCompetitionId'].nunique():,} "
      f"distinct competitions [{time.time()-t0:.0f}s]")

## Five venues, ranked by the metric I was actually using: top-20 median

In [ ]:
MY_VENUES = {
    "playground-series-s6e8": "S6E8",
    "kaggriculture": "Kaggriculture",
    "biohub-cell-tracking-during-development": "Biohub",
    "rsna-knee-abnormality-detection": "RSNA",
    "pokemon-tcg-ai-battle-challenge-strategy": "Pokemon",
}

def venue_votes(slug):
    return merged.loc[merged["Slug"] == slug, "TotalVotes"].dropna().sort_values(ascending=False).to_numpy()

rows = []
for slug, label in MY_VENUES.items():
    v = venue_votes(slug)
    if len(v) == 0:
        continue
    rows.append({
        "venue": label, "n_notebooks": len(v),
        "top20_median": np.median(v[:20]) if len(v) >= 20 else np.median(v),
        "top20_pct_of_venue": round(100 * 20 / len(v), 1),
    })
df = pd.DataFrame(rows).sort_values("top20_median", ascending=False)
print(df.to_string(index=False))

By this metric -- the one this whole portfolio has used for weeks to decide where to publish --
Biohub wins clearly. Look at the third column though: "top 20" is Biohub's most selective **3.1%**
of notebooks, but Kaggriculture's much less selective **7.2%**, and Pokemon's barely-selective
**24.7%**. A fixed count of 20 samples a different depth of each venue's distribution depending on
how many notebooks that venue has. That is not a fair comparison, independent of whatever the true
underlying quality gap is.

## Fix it: compare the same percentile, not the same count

In [ ]:
rows2 = []
for slug, label in MY_VENUES.items():
    v = venue_votes(slug)
    if len(v) == 0:
        continue
    n = len(v)
    k5 = max(1, round(n * 0.05))
    k10 = max(1, round(n * 0.10))
    rows2.append({
        "venue": label, "n_notebooks": n,
        "top5pct_median": round(float(np.median(v[:k5])), 1),
        "top10pct_median": round(float(np.median(v[:k10])), 1),
        "mean": round(float(v.mean()), 2),
        "p75": round(float(np.percentile(v, 75)), 1),
        "p90": round(float(np.percentile(v, 90)), 1),
    })
df2 = pd.DataFrame(rows2)
print(df2.sort_values("top5pct_median", ascending=False).to_string(index=False))
print()
print("Rank by top-5%-median:", " > ".join(df2.sort_values("top5pct_median", ascending=False)["venue"]))
print("Rank by mean:         ", " > ".join(df2.sort_values("mean", ascending=False)["venue"]))

Once matched to the same percentile depth, **Biohub's #1 spot disappears.** Kaggriculture leads on
top-5%-median, top-10%-median, mean, p75, and p90 -- every statistic except the raw population
median (already shown to be flat and near-noise-level, ~1-3 votes, across all five in the
companion notebook).

## Is the mean just one 698-vote notebook in Kaggriculture?

In [ ]:
def trimmed_means(slug):
    v = venue_votes(slug)
    n = len(v)
    k1pct = max(1, round(n * 0.01))
    return {
        "mean_full": round(v.mean(), 2),
        "mean_drop_top1": round(v[1:].mean(), 2) if n > 1 else None,
        "mean_drop_top5": round(v[5:].mean(), 2) if n > 5 else None,
        "mean_drop_top1pct": round(v[k1pct:].mean(), 2) if n > k1pct else None,
        "max_share_of_total_votes_pct": round(100 * v[0] / v.sum(), 1) if v.sum() else None,
    }

for slug, label in [("kaggriculture", "Kaggriculture"), ("biohub-cell-tracking-during-development", "Biohub")]:
    t = trimmed_means(slug)
    print(f"{label:15s} {t}")

No. Kaggriculture's single biggest notebook holds under 17% of the venue's total votes, and the
mean ranking survives dropping the top 1, top 5, or top 1% of notebooks entirely -- Kaggriculture
stays ahead of Biohub's equivalent trimmed mean at every cut. This is not one outlier carrying the
whole comparison.

**Convergent evidence, not just this one method.** A separate measurement of mine from 12 days
earlier, using a completely different method -- a live 14-day recent-notebook exposure proxy
rather than this full population census -- found the same direction: Kaggriculture notebooks
averaged 13.85 displayed votes versus Biohub's 5.85 in that window (2.37x). I designed a controlled
venue-switch test for it at the time and never actually ran it. Publishing this notebook into
Kaggriculture is that test, twelve days late.

## Check your own two venues

In [ ]:
VENUE_A = "kaggriculture"       # <-- replace with any competition slug
VENUE_B = "biohub-cell-tracking-during-development"  # <-- replace with any competition slug

for slug in (VENUE_A, VENUE_B):
    v = venue_votes(slug)
    if len(v) == 0:
        print(f"{slug}: no competition-attributed kernels found in this snapshot")
        continue
    n = len(v)
    k5 = max(1, round(n * 0.05))
    print(f"{slug}: n={n}  mean={v.mean():.2f}  top-5%-median={np.median(v[:k5]):.1f}  "
          f"top-20-median={np.median(v[:20]) if n>=20 else np.median(v):.1f} (={100*min(20,n)/n:.1f}% of venue)")

## The honest caveats

This is a population correction, not a controlled experiment: it does not separate venue from
topic, author reputation, or timing, and a competition's notebook feed is not a random sample of
who might publish there. `CurrentKernelVersionId` attribution is a same-day snapshot that runs
4-8% under a live CLI pagination count for the same competition -- expected drift, not a bug.

And this doesn't mean "go publish in Kaggriculture and expect Biohub's old top-20 number." My own
best notebook across five competitions has taken 6 votes; none of my notebooks have reached any
venue's actual top 5%. The population statistics above describe the venue, not a guarantee for one
more notebook from an author without existing audience there -- mean and top-percentile medians
are pulled up by authors who already have one. What this notebook does settle: if you are choosing
between two venues using each one's "top 20" number, you are not comparing like with like, and the
venue that number favors may not be the venue that actually pays better once you look at people
without a built-in audience, and Kaggriculture would win a broad tier-by-tier accounting.